In [1]:
import threading
from flask import Flask, render_template_string
from flask_sqlalchemy import SQLAlchemy
from datetime import datetime

app = Flask(__name__)

# Tam fərqli verilənlər bazası faylı və YENİ PORT: 8500
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///edu_portal_advanced_8500.db'
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False

db = SQLAlchemy(app)

# 1. Model: Universitetlərin Ümumi Statistikası
class University(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    name = db.Column(db.String(100), nullable=False)
    student_count = db.Column(db.Integer)
    tuition_fee = db.Column(db.String(50)) 
    ip_range = db.Column(db.String(50))    
    type = db.Column(db.String(20))       

# 2. Model: Genişləndirilmiş Giriş Loqları
class EduLog(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    ip_address = db.Column(db.String(50))
    university_name = db.Column(db.String(100))
    user_role = db.Column(db.String(30))       
    activity = db.Column(db.String(100))       
    os_device = db.Column(db.String(50))       
    timestamp = db.Column(db.DateTime, default=datetime.utcnow)

# FRONTEND DİZAYNI (Həqiqi Təhsil Portalı Görünüşü)
html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>Təhsil Portalı - Universitet Reyestri</title>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 0; padding: 25px; background-color: #f4f6f9; color: #333; }
        h2 { color: #1a365d; margin-bottom: 5px; font-size: 26px; display: flex; align-items: center; gap: 10px; }
        .subtitle { color: #7f8c8d; margin-bottom: 30px; font-size: 15px; }
        
        /* Statistik Kartlar */
        .stats-container { display: flex; gap: 20px; margin-bottom: 30px; }
        .card { flex: 1; background: white; padding: 20px; border-radius: 12px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); border-top: 4px solid #2b6cb0; }
        .card.purple { border-top-color: #805ad5; }
        .card.orange { border-top-color: #dd6b20; }
        .card h3 { margin: 0; color: #718096; font-size: 12px; text-transform: uppercase; letter-spacing: 0.5px; }
        .card p { margin: 10px 0 0 0; font-size: 24px; font-weight: bold; color: #2d3748; }
        
        /* Bölmələr */
        .section-title { color: #2b6cb0; font-size: 18px; margin-top: 30px; margin-bottom: 15px; border-left: 4px solid #2b6cb0; padding-left: 10px; }
        .table-container { background: white; padding: 20px; border-radius: 12px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); margin-bottom: 30px; }
        
        /* Cədvəl Strukturları */
        table { border-collapse: collapse; width: 100%; text-align: left; }
        th, td { padding: 12px 15px; border-bottom: 1px solid #e2e8f0; font-size: 14px; }
        th { background-color: #edf2f7; color: #4a5568; font-weight: 600; font-size: 13px; text-transform: uppercase; }
        tr:hover { background-color: #f7fafc; }
        
        /* Axtarış Qutusu */
        .search-box { width: 100%; padding: 12px 15px; border: 1px solid #cbd5e0; border-radius: 8px; font-size: 14px; outline: none; margin-bottom: 20px; box-sizing: border-box; }
        .search-box:focus { border-color: #2b6cb0; box-shadow: 0 0 0 3px rgba(66,153,225,0.2); }
        
        /* Rəngli Badgelər */
        .badge { padding: 4px 8px; border-radius: 6px; font-size: 12px; font-weight: bold; }
        .badge-student { background-color: #ebf8ff; color: #2b6cb0; }
        .badge-teacher { background-color: #f0fff4; color: #22543d; }
        .badge-admin { background-color: #fff5f5; color: #9b2c2c; }
        .badge-gov { background-color: #e6fffa; color: #234e52; }
        .badge-private { background-color: #faf5ff; color: #553c9a; }
        code { background: #edf2f7; padding: 3px 6px; border-radius: 4px; font-family: monospace; color: #c53030; font-size: 13px; }
    </style>
</head>
<body>

    <h2>🎓 Flask ilə Təhsil Portalının IP-sinin Yığılması Sistemi</h2>
    <div class="subtitle">Azərbaycanın TOP Universitetlərinin daxili şəbəkə simulyasiyası və tələbə/müəllim giriş analitikası</div>

    <!-- 1. Təhsil Statistikaları (Kartlar) -->
    <div class="stats-container">
        <div class="card">
            <h3>Sistemdəki Universitet Sayı</h3>
            <p>10 Ali Müəssisə</p>
        </div>
        <div class="card purple">
            <h3>Portal üzrə Ümumi Giriş</h3>
            <p>{{ total_logs }} Aktiv Seans</p>
        </div>
        <div class="card orange">
            <h3>Monitorinq Edilən Tələbə Miqyası</h3>
            <p>~ 120,000+ Tələbə</p>
        </div>
    </div>

    <!-- 2. BÖLMƏ: Top Universitetlərin Reytinq və Qiymət Siyahısı -->
    <div class="section-title">🏛️ Universitetlərin İnformasiya Reyestri və Qiymətləri</div>
    <div class="table-container">
        <table>
            <thead>
                <tr>
                    <th>ID</th>
                    <th>Universitetin Adı</th>
                    <th>Təxmini Tələbə Sayı</th>
                    <th>Təqribi İllik Təhsil Haqqı</th>
                    <th>Təhlükəsiz IP Diapazonu</th>
                    <th>Təsnifat</th>
                </tr>
            </thead>
            <tbody>
                {% for uni in universities %}
                <tr>
                    <td><b>#{{ uni.id }}</b></td>
                    <td><b>{{ uni.name }}</b></td>
                    <td>👤 {{ "{:,}".format(uni.student_count) }} tələbə</td>
                    <td>💰 <code>{{ uni.tuition_fee }} AZN</code></td>
                    <td>🌐 <i>{{ uni.ip_range }}</i></td>
                    <td><span class="badge {% if uni.type=='Dövlət' %}badge-gov{% else %}badge-private{% endif %}">{{ uni.type }}</span></td>
                </tr>
                {% endfor %}
            </tbody>
        </table>
    </div>

    <!-- 3. BÖLMƏ: Real-Time IP Giriş Jurnalı -->
    <div class="section-title">📋 Canlı Təhsil Portalı Giriş Jurnalı (Real-Time Monitor)</div>
    <div class="table-container">
        <input type="text" id="logSearch" class="search-box" onkeyup="filterLogs()" placeholder="Universitet adı, IP, Rol və ya Giriş Məqsədi ilə dinamik axtarış edin...">
        
        <table id="logTable">
            <thead>
                <tr>
                    <th>Giriş ID</th>
                    <th>İstifadəçi IP Adresi</th>
                    <th>Mənsub Olduğu Universitet</th>
                    <th>İstifadəçi Rolu</th>
                    <th>Portalda Etdiyi Əməliyyat (Aktivlik)</th>
                    <th>Cihaz / ƏS</th>
                    <th>Giriş Zamanı (UTC)</th>
                </tr>
            </thead>
            <tbody>
                {% for log in logs %}
                <tr>
                    <td>#{{ log.id }}</td>
                    <td><code>{{ log.ip_address }}</code></td>
                    <td>🏢 {{ log.university_name }}</td>
                    <td>
                        {% if log.user_role == 'Tələbə' %}
                            <span class="badge badge-student">🎓 Tələbə</span>
                        {% elif log.user_role == 'Müəllim' %}
                            <span class="badge badge-teacher">👨‍🏫 Müəllim</span>
                        {% else %}
                            <span class="badge badge-admin">💼 Tyutor / Admin</span>
                        {% endif %}
                    </td>
                    <td style="color: #4a5568; font-weight: 500;">⚡ {{ log.activity }}</td>
                    <td>{{ log.os_device }}</td>
                    <td style="color: #a0aec0;">{{ log.timestamp.strftime('%H:%M:%S') }}</td>
                </tr>
                {% endfor %}
            </tbody>
        </table>
    </div>

    <script>
        function filterLogs() {
            var input = document.getElementById("logSearch");
            var filter = input.value.toUpperCase();
            var table = document.getElementById("logTable");
            var tr = table.getElementsByTagName("tr");

            for (var i = 1; i < tr.length; i++) {
                var showRow = false;
                var tds = tr[i].getElementsByTagName("td");
                for (var j = 0; j < tds.length; j++) {
                    if (tds[j] && tds[j].innerText.toUpperCase().indexOf(filter) > -1) {
                        showRow = true;
                        break;
                    }
                }
                tr[i].style.display = showRow ? "" : "none";
            }
        }
    </script>
</body>
</html>
"""

@app.route('/')
def index():
    with app.app_context():
        universities = db.session.query(University).all()
        logs = db.session.query(EduLog).order_by(EduLog.id.desc()).all()
        total_logs = len(logs)
    return render_template_string(html_content, universities=universities, logs=logs, total_logs=total_logs)

with app.app_context():
    db.create_all()
    
    if db.session.query(University).count() == 0:
        top_universities = [
            University(name="Bakı Dövlət Universiteti (BDU)", student_count=24000, tuition_fee="2000 - 4500", ip_range="10.10.1.*", type="Dövlət"),
            University(name="Azərbaycan Dövlət Neft və Sənaye Universiteti (ADNSU)", student_count=16000, tuition_fee="2100 - 3800", ip_range="10.10.2.*", type="Dövlət"),
            University(name="Azərbaycan Dövlət İqtisad Universiteti (UNEC)", student_count=19000, tuition_fee="2500 - 5000", ip_range="10.10.3.*", type="Dövlət"),
            University(name="ADA Universiteti", student_count=3500, tuition_fee="4000 - 6500", ip_range="10.10.4.*", type="Dövlət"),
            University(name="Bakı Mühəndislik Universiteti (BMU)", student_count=5500, tuition_fee="2200 - 4000", ip_range="10.10.5.*", type="Dövlət"),
            University(name="Xəzər Universiteti", student_count=3000, tuition_fee="3000 - 5500", ip_range="10.10.6.*", type="Özəl"),
            University(name="Qərbi Kaspi Universiteti", student_count=4000, tuition_fee="2000 - 3500", ip_range="10.10.7.*", type="Özəl"),
            University(name="Azərbaycan Dillər Universiteti (ADU)", student_count=8000, tuition_fee="1800 - 3000", ip_range="10.10.8.*", type="Dövlət"),
            University(name="Azərbaycan Tibb Universiteti (ATU)", student_count=9000, tuition_fee="3000 - 5000", ip_range="10.10.9.*", type="Dövlət"),
            University(name="Bakı Ali Neft Məktəbi (BANM)", student_count=1200, tuition_fee="4500 - 5000", ip_range="10.10.10.*", type="Dövlət")
        ]
        db.session.bulk_save_objects(top_universities)
        db.session.commit()

    if db.session.query(EduLog).count() == 0:
        mock_logs = [
            EduLog(ip_address="10.10.1.45", university_name="Bakı Dövlət Universiteti (BDU)", user_role="Tələbə", activity="Onlayn İmtahan Səhifəsi", os_device="Windows 11"),
            EduLog(ip_address="10.10.4.12", university_name="ADA Universiteti", user_role="Müəllim", activity="Elektron Jurnal Girişi", os_device="macOS Sequoia"),
            EduLog(ip_address="10.10.3.112", university_name="Azərbaycan Dövlət İqtisad Universiteti (UNEC)", user_role="Tyutor", activity="Tələbə Qeydiyyatı təsdiqi", os_device="Windows 10"),
            EduLog(ip_address="10.10.2.89", university_name="Azərbaycan Dövlət Neft və Sənaye Universiteti (ADNSU)", user_role="Tələbə", activity="Dərs Materialı Yüklənməsi", os_device="Android Mobile"),
            EduLog(ip_address="10.10.6.23", university_name="Xəzər Universiteti", user_role="Tələbə", activity="Kəsrin Ödənilməsi Portalı", os_device="iPhone iOS")
        ]
        db.session.bulk_save_objects(mock_logs)
        db.session.commit()

def run_flask():
    # Yeni tam təhlükəsiz port: 8500
    app.run(port=8500, debug=False, use_reloader=False)

threading.Thread(target=run_flask, daemon=True).start()
print("YENİ REYESTR PORTALI YENİ PORTDA İŞƏ DÜŞDÜ! Keçid edin: http://127.0.0.1:8500")

YENİ REYESTR PORTALI YENİ PORTDA İŞƏ DÜŞDÜ! Keçid edin: http://127.0.0.1:8500
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:8500
Press CTRL+C to quit
127.0.0.1 - - [18/Jul/2026 18:20:32] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [18/Jul/2026 18:20:32] "GET /favicon.ico HTTP/1.1" 404 -
